# Speckle2Self: Ultrasound Speckle Reduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tue-bmd/zea/blob/main/docs/source/notebooks/models/speckle2self_despeckling_example.ipynb) &nbsp; [![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/tue-bmd/zea/blob/main/docs/source/notebooks/models/speckle2self_despeckling_example.ipynb)

This notebook demonstrates how to perform **self-supervised ultrasound speckle reduction** using [Speckle2Self](https://arxiv.org/abs/2507.06828) within the [zea](https://github.com/tue-bmd/zea) framework.

We apply the model to an **in-vivo carotid artery** scan from [zeahub/zea-carotid-2023](https://huggingface.co/datasets/zeahub/zea-carotid-2023), matching the domain on which the `speckle2self-invivo` weights were trained.

### Speckle2Self
[![arXiv](https://img.shields.io/badge/arXiv-Paper-b31b1b.svg)](https://arxiv.org/abs/2507.06828) &nbsp; [![GitHub](https://img.shields.io/badge/GitHub-Code-black?logo=github)](https://github.com/noseefood/speckle2self)

- Self-supervised: **no clean/noise-free training data required**.
- Works on single-channel ultrasound images.
- Best results on **envelope data** (before log-compression) at ≥ 512 × 512 resolution.

### Workflow
1. Load in-vivo carotid RF data and beamform to linear envelope images (`Beamform` → `EnvelopeDetect`).
2. Load the Speckle2Self Keras model and run inference on the linear envelope data.
3. Log-compress and compare original vs. despeckled B-mode images.


‼️ **Important:** This notebook is optimized for **GPU/TPU**. Code execution on a **CPU** may be very slow.

If you are running in Colab, please enable a hardware accelerator via:

**Runtime → Change runtime type → Hardware accelerator → GPU/TPU** 🚀.

In [1]:
%%capture
%pip install zea

In [2]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt

import zea
from zea.data import load_file
from zea.ops import Beamform, EnvelopeDetect, Pipeline
from zea.visualize import set_mpl_style

zea.init_device(verbose=False)
set_mpl_style()

zea: Using backend 'tensorflow'


## Step 1: Load Carotid Data

We load a few frames of an in-vivo carotid scan from [zeahub/zea-carotid-2023](https://huggingface.co/datasets/zeahub/zea-carotid-2023) on Hugging Face.

The pipeline stops **after envelope detection** (no `Normalize` / `LogCompress`) because Speckle2Self works on **linear-scale envelope data**.


In [3]:
CAROTID_PATH = "hf://zeahub/zea-carotid-2023/2_cross_bifur_right_0000_small.hdf5"
FRAME_IDX = [0, 7, 14]  # number of frames to process
N_TX = 11  # transmits per frame (fewer → faster; more → better quality)

data, scan, probe = load_file(CAROTID_PATH, "raw_data", indices=FRAME_IDX)

scan.set_transmits(N_TX)
scan.zlims = (0, 0.04)
scan.xlims = probe.xlims
scan.n_ch = data.shape[-1]  # RF data: channels in last dim

# Pipeline: beamform + envelope_detect only (no normalize / log_compress)
pipeline = Pipeline(
    operations=[
        Beamform(
            beamformer="delay_and_sum",
            enable_pfield=True,
            num_patches=100,
        ),
        EnvelopeDetect(),
    ],
    with_batch_dim=False,
    jit_options="pipeline",
)
parameters = pipeline.prepare_parameters(probe, scan)
parameters.pop("dynamic_range", None)

n_frames = data.shape[0]
print(f"Loaded {n_frames} frame(s). Raw shape: {data.shape}")
print(f"Scan xlims: {[round(v * 1e3, 1) for v in scan.xlims]} mm")
print(f"Scan zlims: {[round(v * 1e3, 1) for v in scan.zlims]} mm")

zea: WARNING No transmit origins provided, using zeros
zea: Running compute_pfield and caching the result to /root/.cache/zea/cached_funcs/compute_pfield_19f603da395129bbe22dcc0a986382a1.pkl.


zea: Computing pressure field for all transmits
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/transmits
Loaded 3 frame(s). Raw shape: (3, 149, 2176, 128, 1)
Scan xlims: [np.float32(-19.1), np.float32(19.1)] mm
Scan zlims: [0.0, 40.0] mm


In [4]:
n_imgs = n_frames  # process all loaded frames

envelopes = []
for i in range(n_imgs):
    out = pipeline(data=data[i, scan.selected_transmits], **parameters)
    envelopes.append(np.array(out[pipeline.output_key]))  # (H, W)

envelopes = np.stack(envelopes)  # (N, H, W)
print(f"Envelope shape: {envelopes.shape}  range: [{envelopes.min():.3g}, {envelopes.max():.3g}]")

zea: DEBUG [zea.Pipeline] The following input keys are not used by the pipeline: {'zlims', 'xlims', 'center_frequency', 'n_el'}. Make sure this is intended. This warning will only be shown once.


Envelope shape: (3, 812, 774)  range: [0.00568, 5.33e+04]


## Step 2: Load the Speckle2Self Model

`Speckle2Self` is implemented as a **native Keras 3 model** — no ONNX runtime required.
Load it directly from a Hugging Face preset:

```python
model = Speckle2Self.from_preset("speckle2self-invivo")
```

Or from a local `.pth` checkpoint (e.g. for development):

```python
model = Speckle2Self.from_pth("/path/to/model_2833.pth")
```


In [5]:
import os
from zea.models.speckle2self import Speckle2Self

# ── Local preset directory ────────────────────────────────────────────────────
# Point this to a local directory that holds the saved Keras preset
# (config.json + model.weights.h5).
# Once weights are published on Hugging Face, switch to:
#   model = Speckle2Self.from_preset("speckle2self-invivo")
LOCAL_PRESET = "/workspace/speckle2self-local"

# One-time conversion: .pth → Keras preset (requires torch; run once then reuse)
if not os.path.exists(os.path.join(LOCAL_PRESET, "model.weights.h5")):
    PTH_PATH = "/workspace/inVivo/model_2833.pth"
    print(f"Converting {PTH_PATH} → {LOCAL_PRESET}/ ...")
    _tmp = Speckle2Self.from_pth(PTH_PATH)
    _tmp.save_to_preset(LOCAL_PRESET)
    print(f"Saved preset to {LOCAL_PRESET}/")

model = Speckle2Self.from_preset(LOCAL_PRESET)
print("Model loaded:", model)

Model loaded: <Speckle2Self name=speckle2_self, built=True>


## Step 3: Preprocess and Run Inference

The model expects **linear-scale envelope data** as input (not log-compressed):

1. **Normalise** each frame to `[0, 1]` per-image: `(x − min) / (max − min)`.
2. **Shape**: `[N, 1, H, W]` — NCHW, float32.
3. **Output**: despeckled linear envelope in `[0, 1]`.

Both the input and output are then log-compressed for clinical B-mode display.


In [6]:
def linear_normalize(img: np.ndarray) -> np.ndarray:
    """Per-image min–max normalisation to [0, 1]."""
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-16)


def to_bmode(img: np.ndarray, dynamic_range: tuple = (-60, 0)) -> np.ndarray:
    """Log-compress a linear envelope to a display B-mode image.

    Normalises to [0, 1] first, then applies 20 log10, clips to
    ``dynamic_range`` dB, and rescales to [0, 1].
    """
    img_n = linear_normalize(img)
    db = 20 * np.log10(np.clip(img_n, 1e-6, None))
    db = np.clip(db, dynamic_range[0], dynamic_range[1])
    return (db - dynamic_range[0]) / (dynamic_range[1] - dynamic_range[0])


# Per-image normalisation → [N, 1, H, W] model input (matches original inference.py)
envelopes_norm = np.stack([linear_normalize(e) for e in envelopes])  # (N, H, W)
model_input = envelopes_norm[:, np.newaxis, :, :].astype(np.float32)  # (N, 1, H, W)

# Run speckle reduction
despeckled = model(model_input)  # (N, 1, H, W), values in [0, 1]
despeckled = np.array(despeckled)[:, 0]  # (N, H, W)

print(f"Input  {model_input.shape}  mean={envelopes_norm.mean():.3f}")
print(f"Output {despeckled.shape}  mean={despeckled.mean():.3f}  max={despeckled.max():.3f}")

Input  (3, 1, 812, 774)  mean=0.032
Output (3, 812, 774)  mean=0.144  max=0.889


## Step 4: Visualise Results

Both the original and despeckled envelopes are log-compressed with the same pipeline (`to_bmode`), each normalised independently to its own maximum (0 dB).

The despeckled image typically appears brighter: the final decoder layer (`InstanceNorm → ReLU`) centres the output around zero, so background pixels that were near-zero in the linear input are lifted to small positive values. After log compression this raises the apparent noise floor. The clinically relevant observation is the **speckle texture reduction** — smooth regions in the output where the input showed granular speckle.


In [7]:
DYNAMIC_RANGE = (-40, 0)  # dB

xlims_mm = [v * 1e3 for v in scan.xlims]
zlims_mm = [v * 1e3 for v in scan.zlims]
extent = [xlims_mm[0], xlims_mm[1], zlims_mm[1], zlims_mm[0]]

# Both use the same log-compression pipeline; each image is normalised to its own max.
bmode_original = np.stack([to_bmode(e, DYNAMIC_RANGE) for e in envelopes])
# bmode_despeckled = np.stack([to_bmode(d, DYNAMIC_RANGE) for d in despeckled])
bmode_despeckled = despeckled  # already in [0, 1], so skip to_bmode

fig, axes = plt.subplots(2, n_imgs, figsize=(4 * n_imgs, 8), squeeze=False)

for i in range(n_imgs):
    axes[0, i].imshow(bmode_original[i], cmap="gray", vmin=0, vmax=1, extent=extent)
    axes[0, i].set_title(f"Original — frame {i}", fontsize=10)
    axes[0, i].set_xlabel("X (mm)")
    axes[0, i].set_ylabel("Z (mm)")

    axes[1, i].imshow(bmode_despeckled[i], cmap="gray", vmin=0, vmax=1, extent=extent)
    axes[1, i].set_title(f"Despeckled (Speckle2Self) — frame {i}", fontsize=10)
    axes[1, i].set_xlabel("X (mm)")
    axes[1, i].set_ylabel("Z (mm)")

fig.suptitle("Carotid In-Vivo — Speckle Reduction", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("speckle2self_output.png", bbox_inches="tight", dpi=100)
plt.close()

**Speckle2Self results on in-vivo carotid data.**

*Top row*: original B-mode (log-compressed linear envelope).  
*Bottom row*: despeckled — speckle granularity is reduced and vessel boundaries become clearer.

![Image](./speckle2self_output.png)